In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DAILY_DATA_DIR = PROJECT_ROOT / "data" / "daily"
MINUTE_DATA_DIR = PROJECT_ROOT / "data" / "minute"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
RESEARCH_DIR = PROJECT_ROOT / "research"

PROCESSED_DATA_DIR = OUTPUTS_DIR / "processed_data"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
RESEARCH_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook directory:", NOTEBOOK_DIR)
print("Project root:", PROJECT_ROOT)
print("Daily data directory:", DAILY_DATA_DIR)
print("Minute data directory:", MINUTE_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)

In [ ]:
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
assert DAILY_DATA_DIR.exists(), f"Daily data directory not found: {DAILY_DATA_DIR}"
assert MINUTE_DATA_DIR.exists(), f"Minute data directory not found: {MINUTE_DATA_DIR}"

print("All required directories were found.")

In [ ]:
daily_files = sorted(DAILY_DATA_DIR.glob("*.parquet"))

print("Number of daily parquet files:", len(daily_files))

for file_path in daily_files[:10]:
    print(file_path.name)

In [ ]:
EXPECTED_SYMBOL_COUNT = 208

if len(daily_files) == EXPECTED_SYMBOL_COUNT:
    print("Daily file count matches the assignment.")
else:
    print(
        f"Warning: expected {EXPECTED_SYMBOL_COUNT} files, "
        f"but found {len(daily_files)}."
    )

In [ ]:
sample_daily_file = daily_files[0]
sample_symbol = sample_daily_file.stem

sample_daily_df = pd.read_parquet(sample_daily_file)

print("Sample file:", sample_daily_file.name)
print("Symbol:", sample_symbol)
print("Shape:", sample_daily_df.shape)

print(sample_daily_df.head())
print("\n" * 2)
print(sample_daily_df.info())


In [ ]:
EXPECTED_DAILY_COLUMNS = [
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume",
]

print("Columns found:", sample_daily_df.columns.tolist())

missing_columns = set(EXPECTED_DAILY_COLUMNS) - set(sample_daily_df.columns)

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("Expected daily columns are present.")

In [ ]:
def load_daily_file(file_path: Path) -> pd.DataFrame:
    """
    Load one stock's daily parquet file and add its symbol.

    Parameters
    ----------
    file_path : Path
        Path to the parquet file.

    Returns
    -------
    pd.DataFrame
        Clean daily OHLCV data with a symbol column.
    """
    df = pd.read_parquet(file_path).copy()

    df.columns = [str(column).strip().lower() 
                  for column in df.columns
                  ]

    required_columns = {"date", "open", "high", "low", "close", "volume",}

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{file_path.name} is missing columns: "
            f"{sorted(missing_columns)}"
        )

    df = df[["date","open","high","low","close","volume",]].copy()

    df["date"] = pd.to_datetime(df["date"], errors="raise",)

    df["symbol"] = file_path.stem

    return df

In [ ]:
test_daily_df = load_daily_file(sample_daily_file)

print(test_daily_df.shape)
test_daily_df.head()

In [ ]:
daily_frames = []

for file_path in daily_files:
    stock_daily_df = load_daily_file(file_path)
    daily_frames.append(stock_daily_df)

daily_df = pd.concat(daily_frames,axis=0, ignore_index=True,)

daily_df = daily_df.sort_values(["symbol", "date"]).reset_index(drop=True)

daily_df = daily_df[["date", "symbol", "open", "high", "low", "close", "volume", ]]

print("Combined daily dataset shape:", daily_df.shape)
daily_df.head()

In [ ]:
print("Rows:", len(daily_df))
print("Symbols:", daily_df["symbol"].nunique())
print("Trading dates:", daily_df["date"].nunique())
print("Date range:", daily_df["date"].min(), "to", daily_df["date"].max(),)

In [ ]:
duplicate_mask = daily_df.duplicated(subset=["symbol", "date"],keep=False,)

duplicate_count = int(duplicate_mask.sum())

print("Duplicate symbol-date rows:", duplicate_count)

if duplicate_count > 0:
    display(
        daily_df.loc[duplicate_mask]
        .sort_values(["symbol", "date"])
        .head(20)
    )

In [ ]:
missing_summary = daily_df.isna().sum().to_frame(name="missing_count")
missing_summary["missing_pct"] = (missing_summary["missing_count"]/ len(daily_df)* 100)
missing_summary

In [ ]:
quality_checks = pd.Series(
    {
        "non_positive_open": (daily_df["open"] <= 0).sum(),
        "non_positive_high": (daily_df["high"] <= 0).sum(),
        "non_positive_low": (daily_df["low"] <= 0).sum(),
        "non_positive_close": (daily_df["close"] <= 0).sum(),
        "negative_volume": (daily_df["volume"] < 0).sum(),
        "high_below_low": (
            daily_df["high"] < daily_df["low"]
        ).sum(),
        "high_below_open": (
            daily_df["high"] < daily_df["open"]
        ).sum(),
        "high_below_close": (
            daily_df["high"] < daily_df["close"]
        ).sum(),
        "low_above_open": (
            daily_df["low"] > daily_df["open"]
        ).sum(),
        "low_above_close": (
            daily_df["low"] > daily_df["close"]
        ).sum(),
    },
    name="count",
)

quality_checks

In [ ]:
rows_per_symbol = (daily_df.groupby("symbol").size().sort_values())

rows_per_symbol.describe()

In [ ]:
rows_per_symbol.head(20)

In [ ]:
symbols_per_date = (daily_df.groupby("date")["symbol"].nunique().sort_values())

symbols_per_date.head(10)

In [ ]:
daily_panel_path = (PROCESSED_DATA_DIR / "daily_panel_raw.parquet")

daily_df.to_parquet(daily_panel_path, index=False,)

print("Saved combined daily panel to:")
print(daily_panel_path)

In [ ]:
daily_df.head()

# Target Construction


In [ ]:
target_df = daily_df.copy()

target_df = target_df.sort_values(["symbol", "date"]).reset_index(drop=True)

print(target_df.shape)
target_df.head()

In [ ]:
symbol_groups = target_df.groupby("symbol",sort=False,)

target_df["target_date"] = (symbol_groups["date"].shift(-1))

target_df["next_open"] = (symbol_groups["open"].shift(-1))

In [ ]:
target_df["actual_return_pct"] = ((target_df["next_open"]/ target_df["close"])- 1) * 100

target_df["actual_direction"] = np.where(target_df["actual_return_pct"] >= 0,1,-1,)

target_df["actual_magnitude_pct"] = (target_df["actual_return_pct"].abs())

In [ ]:
target_df["calendar_gap_days"] = (target_df["target_date"]- target_df["date"]).dt.days

In [ ]:
target_columns = ["date","target_date","symbol","close","next_open","actual_return_pct","actual_direction","actual_magnitude_pct","calendar_gap_days",]

target_df[target_columns].head(5)

In [ ]:
sample_symbol = "RELIANCE"

target_df.loc[target_df["symbol"] == sample_symbol, target_columns,].head(5)

In [ ]:
valid_target_mask = target_df["target_date"].notna()

invalid_target_dates = (target_df.loc[valid_target_mask, "target_date"] <= target_df.loc[valid_target_mask, "date"]).sum()

print("Rows where target_date is not after date:",invalid_target_dates,)

In [ ]:
target_df["actual_direction"].value_counts(dropna=False)

In [ ]:
missing_target_mask = target_df["actual_return_pct"].isna()

target_df.loc[missing_target_mask,"actual_direction",] = np.nan

In [ ]:
target_df["actual_direction"].value_counts(dropna=False)

In [ ]:
negative_magnitude_count = (target_df["actual_magnitude_pct"] < 0).sum()

print("Negative magnitude rows:",negative_magnitude_count,)

In [ ]:
target_df["calendar_gap_days"].value_counts(dropna=False).sort_index()

In [ ]:
missing_target_summary = target_df[["target_date","next_open","actual_return_pct","actual_direction","actual_magnitude_pct",]].isna().sum()

missing_target_summary

In [ ]:
daily_with_targets_full = target_df.copy()


daily_with_targets = (
    target_df
    .dropna(
        subset=[
            "target_date",
            "next_open",
            "actual_return_pct",
            "actual_direction",
            "actual_magnitude_pct",
        ]
    )
    .copy()
)

daily_with_targets["actual_direction"] = (daily_with_targets["actual_direction"].astype("int8"))

print("Rows before dropping missing targets:",len(target_df),)

print("Rows after dropping missing targets:",len(daily_with_targets),)

print("Rows removed:",len(target_df) - len(daily_with_targets),)

In [ ]:
daily_with_targets = daily_with_targets.rename(columns={"date": "pred_date"})


daily_with_targets[["pred_date","target_date","symbol","close","next_open","actual_return_pct","actual_direction","actual_magnitude_pct","calendar_gap_days",]].head()

In [ ]:
assert not daily_with_targets[["pred_date","target_date","symbol","actual_return_pct","actual_direction","actual_magnitude_pct",]].isna().any().any()

assert (daily_with_targets["target_date"] > daily_with_targets["pred_date"]).all()

assert daily_with_targets["actual_direction"].isin([-1, 1]).all()

assert (daily_with_targets["actual_magnitude_pct"] >= 0).all()

assert np.allclose(daily_with_targets["actual_magnitude_pct"],daily_with_targets["actual_return_pct"].abs(),)

print("All target-construction checks passed.")

In [ ]:
target_panel_path = (PROCESSED_DATA_DIR/ "daily_panel_with_targets_v1.parquet")

daily_with_targets.to_parquet(target_panel_path,index=False,)

print("Saved target panel to:", target_panel_path)

In [ ]:
daily_with_targets.head()



### Daily Feature Engineering — Version 1

Daily features are calculated separately within each stock.

All features for prediction date `T` use only information available by the close of `T`.

Historical overnight features are shifted by one row because `actual_return_pct`
for row `T` is only realised at the open of `target_date`.

In [ ]:
daily_features_df = daily_with_targets.copy()

daily_features_df = daily_features_df.sort_values(["symbol", "pred_date"]).reset_index(drop=True)

print("Starting shape:", daily_features_df.shape)
daily_features_df.head()

In [ ]:
symbol_groups = daily_features_df.groupby("symbol",sort=False,group_keys=False,)

In [ ]:
# The current row's actual_return_pct belongs to the future target.
# Shift by one so the feature contains only previously realised gaps.

daily_features_df["lagged_overnight_return_1d"] = (symbol_groups["actual_return_pct"].shift(1))

daily_features_df["gap_mean_20d"] = (daily_features_df.groupby("symbol")["lagged_overnight_return_1d"].transform(lambda series: series.rolling(window=20,min_periods=20,).mean()))

daily_features_df["overnight_std_20d"] = (daily_features_df.groupby("symbol")["lagged_overnight_return_1d"].transform(lambda series: series.rolling(window=20,min_periods=20,).std()))

daily_features_df["gap_positive_fraction_20d"] = (daily_features_df.groupby("symbol")["lagged_overnight_return_1d"].transform(lambda series: (series.ge(0).where(series.notna()).rolling(window=20,min_periods=20,).mean())))

In [ ]:
overnight_feature_columns = ["pred_date","target_date","symbol","actual_return_pct","lagged_overnight_return_1d","gap_mean_20d","overnight_std_20d","gap_positive_fraction_20d",]

daily_features_df.loc[daily_features_df["symbol"] == "RELIANCE",overnight_feature_columns,].head(25)

In [ ]:
daily_features_df["return_1d"] = (symbol_groups["close"].pct_change(1) * 100)

daily_features_df["return_5d"] = (symbol_groups["close"].pct_change(5) * 100)

daily_features_df["return_20d"] = (symbol_groups["close"].pct_change(20) * 100)

In [ ]:
daily_features_df["daily_volatility_5d"] = (daily_features_df.groupby("symbol")["return_1d"].transform(lambda series: series.rolling(window=5,min_periods=5,).std()))

daily_features_df["daily_volatility_20d"] = (daily_features_df.groupby("symbol")["return_1d"].transform(lambda series: series.rolling(window=20,min_periods=20,).std()))

In [ ]:
daily_features_df["volume_mean_20d_lagged"] = (symbol_groups["volume"].transform(lambda series: (series.shift(1).rolling(window=20,min_periods=20,).mean())))

daily_features_df["volume_std_20d_lagged"] = (symbol_groups["volume"].transform(lambda series: (series.shift(1).rolling(window=20,min_periods=20,).std())))

daily_features_df["volume_zscore_20d"] = ((daily_features_df["volume"]- daily_features_df["volume_mean_20d_lagged"])/ daily_features_df["volume_std_20d_lagged"])

In [ ]:
daily_features_df["volume_mean_5d"] = (symbol_groups["volume"].transform(lambda series: series.rolling(window=5,min_periods=5,).mean()))

daily_features_df["volume_mean_20d"] = (symbol_groups["volume"].transform(lambda series: series.rolling(window=20,min_periods=20,).mean()))

daily_features_df["volume_trend_5d_20d"] = (daily_features_df["volume_mean_5d"] / daily_features_df["volume_mean_20d"])

In [ ]:
daily_features_df["day_of_week"] = (daily_features_df["pred_date"].dt.dayofweek)

In [ ]:
daily_feature_columns = ["lagged_overnight_return_1d","return_1d","return_5d","return_20d","daily_volatility_20d","overnight_std_20d","daily_volatility_5d","gap_mean_20d","gap_positive_fraction_20d","volume_zscore_20d","volume_trend_5d_20d","calendar_gap_days","day_of_week",]

print("Number of unique Version 1 daily features:", len(daily_feature_columns))

In [ ]:
daily_feature_missingness = daily_features_df[daily_feature_columns].isna().sum().to_frame("missing_count")

daily_feature_missingness["missing_pct"] = (daily_feature_missingness["missing_count"] / len(daily_features_df) * 100)

daily_feature_missingness.sort_values("missing_pct", ascending=False)

In [ ]:
daily_feature_infinite_counts = pd.Series({column: np.isinf(daily_features_df[column]).sum() for column in daily_feature_columns},name="infinite_count",)

daily_feature_infinite_counts

In [ ]:
daily_features_df[daily_feature_columns] = (daily_features_df[daily_feature_columns].replace([np.inf, -np.inf], np.nan))

In [ ]:
assert daily_features_df["day_of_week"].between(0, 6).all()

assert (daily_features_df["calendar_gap_days"] > 0).all()

valid_positive_fraction = daily_features_df["gap_positive_fraction_20d"].dropna()

assert valid_positive_fraction.between(0, 1).all()

print("Daily feature sanity checks passed.")

In [ ]:
sample_columns = ["pred_date","symbol","lagged_overnight_return_1d","gap_mean_20d","overnight_std_20d","gap_positive_fraction_20d","return_1d","return_5d","return_20d","daily_volatility_5d","daily_volatility_20d","volume_zscore_20d","volume_trend_5d_20d","calendar_gap_days","day_of_week","actual_return_pct",]

daily_features_df.loc[daily_features_df["symbol"] == "RELIANCE",sample_columns,].tail(10)

In [ ]:
daily_features_df[daily_feature_columns].describe().T

In [ ]:
feature_correlation = daily_features_df[daily_feature_columns].corr()

feature_correlation

In [ ]:
usable_rows = daily_features_df.dropna(subset=daily_feature_columns)

print("Total rows:", len(daily_features_df))
print("Rows with all daily features available:", len(usable_rows))
print("Coverage: {:.2f}%".format(len(usable_rows) / len(daily_features_df) * 100))

In [ ]:
pd.DataFrame(
{
        "missing": daily_features_df[daily_feature_columns].isna().sum(),
        "missing_pct": daily_features_df[daily_feature_columns].isna().mean() * 100,
    }
).sort_values("missing_pct", ascending=False)

In [ ]:
daily_features_path = PROCESSED_DATA_DIR / "daily_features_v1.parquet"

daily_features_df.to_parquet(daily_features_path,index=False,)

print("Saved daily feature panel to:")
print(daily_features_path)

### Minute Feature Engineering

In [ ]:
minute_files = sorted(MINUTE_DATA_DIR.glob("*.parquet"))

print("Number of minute parquet files:", len(minute_files))

for file_path in minute_files[:5]: 
    print(file_path.name)

In [ ]:
daily_symbols = {file_path.stem for file_path in daily_files}
minute_symbols = {file_path.stem for file_path in minute_files}

print("Daily symbols:", len(daily_symbols))
print("Minute symbols:", len(minute_symbols))
print("Present only in daily:", sorted(daily_symbols - minute_symbols))
print("Present only in minute:", sorted(minute_symbols - daily_symbols))

In [ ]:
sample_minute_file = MINUTE_DATA_DIR / "RELIANCE.parquet"

if not sample_minute_file.exists():
    sample_minute_file = minute_files[0]

sample_minute_symbol = sample_minute_file.stem

print("Sample file:", sample_minute_file.name)
print("Sample symbol:", sample_minute_symbol)

In [ ]:
sample_minute_df = pd.read_parquet(sample_minute_file).copy()

print("Shape:", sample_minute_df.shape)

sample_minute_df.head()

In [ ]:
print("Columns:", sample_minute_df.columns.tolist())

sample_minute_df.info()

In [ ]:
sample_minute_df.columns = [str(column).strip().lower() for column in sample_minute_df.columns]

sample_minute_df["timestamp"] = pd.to_datetime(sample_minute_df["timestamp"], errors="raise")

sample_minute_df = sample_minute_df.sort_values("timestamp").reset_index(drop=True)

sample_minute_df["date"] = sample_minute_df["timestamp"].dt.normalize()
sample_minute_df["time"] = sample_minute_df["timestamp"].dt.time
sample_minute_df["symbol"] = sample_minute_symbol

sample_minute_df.head()

In [ ]:
print("First timestamp:", sample_minute_df["timestamp"].min())
print("Last timestamp:", sample_minute_df["timestamp"].max())
print("Number of trading dates:", sample_minute_df["date"].nunique())

In [ ]:
duplicate_timestamp_count = sample_minute_df.duplicated(subset=["timestamp"]).sum()

print("Duplicate timestamps:", duplicate_timestamp_count)

In [ ]:
sample_minute_missingness = sample_minute_df.isna().sum().to_frame("missing_count")

sample_minute_missingness["missing_pct"] = (sample_minute_missingness["missing_count"] / len(sample_minute_df) * 100)

sample_minute_missingness

In [ ]:
print("Earliest observed bar time:", sample_minute_df["time"].min())
print("Latest observed bar time:", sample_minute_df["time"].max())

In [ ]:
bars_per_day = sample_minute_df.groupby("date").size()

bars_per_day.describe()

In [ ]:
bars_per_day.value_counts().sort_index()

In [ ]:
incomplete_session_dates = bars_per_day[bars_per_day != 375]

print("Incomplete sessions:", len(incomplete_session_dates))

incomplete_session_dates.head(10)

In [ ]:
complete_dates = bars_per_day[bars_per_day == 375].index

sample_complete_date = complete_dates[0]

sample_complete_session = sample_minute_df.loc[sample_minute_df["date"] == sample_complete_date].copy()

print("Sample complete date:", sample_complete_date)
print("Bars:", len(sample_complete_session))
print("First timestamp:", sample_complete_session["timestamp"].min())
print("Last timestamp:", sample_complete_session["timestamp"].max())

sample_complete_session.head()

In [ ]:
sample_complete_session["minute_return"] = sample_complete_session["close"].pct_change()

In [ ]:
intraday_realized_volatility = np.sqrt(np.square(sample_complete_session["minute_return"].dropna()).sum())

intraday_realized_volatility

In [ ]:
session_open = sample_complete_session["open"].iloc[0]
session_close = sample_complete_session["close"].iloc[-1]

full_session_return = session_close / session_open - 1

print("Session open:", session_open)
print("Session close:", session_close)
print("Full-session return:", full_session_return)

In [ ]:
last_30_minute_rows = sample_complete_session.tail(30).copy()

print("First bar in final 30 minutes:", last_30_minute_rows["timestamp"].min())
print("Last bar in final 30 minutes:", last_30_minute_rows["timestamp"].max())
print("Rows:", len(last_30_minute_rows))

In [ ]:
last_30_start_price = last_30_minute_rows["open"].iloc[0]
last_30_end_price = last_30_minute_rows["close"].iloc[-1]

last_30_minute_return = last_30_end_price / last_30_start_price - 1

print("Final 30-minute return:", last_30_minute_return)

In [ ]:
if abs(full_session_return) > 1e-12:
    close_auction_return_concentration = last_30_minute_return / full_session_return
else:
    close_auction_return_concentration = np.nan

close_auction_return_concentration

In [ ]:
full_session_volume = sample_complete_session["volume"].sum()
last_30_minute_volume = last_30_minute_rows["volume"].sum()

if full_session_volume > 0:
    close_auction_volume_concentration = last_30_minute_volume / full_session_volume
else:
    close_auction_volume_concentration = np.nan

close_auction_volume_concentration

In [ ]:
session_midpoint = len(sample_complete_session) // 2

morning_session = sample_complete_session.iloc[:session_midpoint].copy()
afternoon_session = sample_complete_session.iloc[session_midpoint:].copy()

print("Morning bars:", len(morning_session))
print("Afternoon bars:", len(afternoon_session))

In [ ]:
morning_return = (morning_session["close"].iloc[-1] / morning_session["open"].iloc[0] - 1)

afternoon_return = (afternoon_session["close"].iloc[-1] / afternoon_session["open"].iloc[0] - 1)

morning_vs_afternoon_return = morning_return - afternoon_return

print("Morning return:", morning_return)
print("Afternoon return:", afternoon_return)
print("Morning minus afternoon:", morning_vs_afternoon_return)

In [ ]:
session_typical_price = (sample_complete_session["high"]+ sample_complete_session["low"]+ sample_complete_session["close"]) / 3

session_vwap_numerator = (session_typical_price * sample_complete_session["volume"]).sum()

session_vwap_denominator = sample_complete_session["volume"].sum()

if session_vwap_denominator > 0:
    session_vwap = session_vwap_numerator / session_vwap_denominator
else:
    session_vwap = np.nan

session_vwap

In [ ]:
if pd.notna(session_vwap) and session_vwap != 0:
    close_vwap_deviation = session_close / session_vwap - 1
else:
    close_vwap_deviation = np.nan

close_vwap_deviation

In [ ]:
valid_amihud_rows = sample_complete_session[
    sample_complete_session["minute_return"].notna()
    & (sample_complete_session["volume"] > 0)
].copy()

valid_amihud_rows["amihud_component"] = (
    valid_amihud_rows["minute_return"].abs()
    / valid_amihud_rows["volume"]
)

amihud_illiquidity = valid_amihud_rows["amihud_component"].mean()

amihud_illiquidity

In [ ]:
sample_minute_features = pd.Series(
    {
        "intraday_realized_volatility_pct": intraday_realized_volatility * 100,
        "close_auction_return_concentration": close_auction_return_concentration,
        "close_auction_volume_concentration": close_auction_volume_concentration,
        "morning_vs_afternoon_return_pct": morning_vs_afternoon_return * 100,
        "close_vwap_deviation_pct": close_vwap_deviation * 100,
        "amihud_illiquidity": amihud_illiquidity,
        "minute_bar_count": len(sample_complete_session),
        "is_complete_session": len(sample_complete_session) == 375,
    }
)

sample_minute_features

### Build Reusable Minute-to-Daily Feature Function

The following function converts one stock's raw minute bars into one row per trading date.

In [ ]:
def create_minute_daily_features(df: pd.DataFrame, symbol: str) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(column).strip().lower() for column in df.columns]
    required_columns = {"timestamp", "open", "high", "low", "close", "volume"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{symbol} minute data is missing columns: {sorted(missing_columns)}"
        )

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise")

    df = df.sort_values("timestamp").reset_index(drop=True)

    df["pred_date"] = df["timestamp"].dt.normalize()

    daily_rows = []

    for pred_date, session_df in df.groupby("pred_date", sort=True):
        session_df = session_df.sort_values("timestamp").reset_index(drop=True)

        minute_bar_count = len(session_df)
        is_complete_session = minute_bar_count == 375

        session_df["minute_return"] = session_df["close"].pct_change()

        valid_minute_returns = session_df["minute_return"].dropna()

        if len(valid_minute_returns) > 0:
            intraday_realized_volatility_pct = (
                np.sqrt(np.square(valid_minute_returns).sum()) * 100
            )
        else:
            intraday_realized_volatility_pct = np.nan

        session_open = session_df["open"].iloc[0]
        session_close = session_df["close"].iloc[-1]

        if session_open > 0:
            full_session_return = session_close / session_open - 1
        else:
            full_session_return = np.nan

        last_30_rows = session_df.tail(min(30, minute_bar_count)).copy()

        if len(last_30_rows) > 0 and last_30_rows["open"].iloc[0] > 0:
            last_30_return = (
                last_30_rows["close"].iloc[-1]
                / last_30_rows["open"].iloc[0]
                - 1
            )
        else:
            last_30_return = np.nan

        if pd.notna(full_session_return) and abs(full_session_return) > 1e-12:
            close_auction_return_concentration = (
                last_30_return / full_session_return
            )
        else:
            close_auction_return_concentration = np.nan

        full_session_volume = session_df["volume"].sum()
        last_30_volume = last_30_rows["volume"].sum()

        if full_session_volume > 0:
            close_auction_volume_concentration = (
                last_30_volume / full_session_volume
            )
        else:
            close_auction_volume_concentration = np.nan

        session_midpoint = minute_bar_count // 2

        morning_session = session_df.iloc[:session_midpoint]
        afternoon_session = session_df.iloc[session_midpoint:]

        if (
            len(morning_session) > 0
            and morning_session["open"].iloc[0] > 0
        ):
            morning_return = (
                morning_session["close"].iloc[-1]
                / morning_session["open"].iloc[0]
                - 1
            )
        else:
            morning_return = np.nan

        if (
            len(afternoon_session) > 0
            and afternoon_session["open"].iloc[0] > 0
        ):
            afternoon_return = (
                afternoon_session["close"].iloc[-1]
                / afternoon_session["open"].iloc[0]
                - 1
            )
        else:
            afternoon_return = np.nan

        if pd.notna(morning_return) and pd.notna(afternoon_return):
            morning_vs_afternoon_return_pct = (
                morning_return - afternoon_return
            ) * 100
        else:
            morning_vs_afternoon_return_pct = np.nan

        typical_price = (
            session_df["high"]
            + session_df["low"]
            + session_df["close"]
        ) / 3

        vwap_denominator = session_df["volume"].sum()

        if vwap_denominator > 0:
            session_vwap = (
                typical_price * session_df["volume"]
            ).sum() / vwap_denominator
        else:
            session_vwap = np.nan

        if pd.notna(session_vwap) and session_vwap != 0:
            close_vwap_deviation_pct = (
                session_close / session_vwap - 1
            ) * 100
        else:
            close_vwap_deviation_pct = np.nan

        valid_amihud_rows = session_df[
            session_df["minute_return"].notna()
            & (session_df["volume"] > 0)
        ]

        if len(valid_amihud_rows) > 0:
            amihud_illiquidity = (
                valid_amihud_rows["minute_return"].abs()
                / valid_amihud_rows["volume"]
            ).mean()
        else:
            amihud_illiquidity = np.nan

        daily_rows.append(
            {
                "pred_date": pred_date,
                "symbol": symbol,
                "intraday_realized_volatility_pct": intraday_realized_volatility_pct,
                "close_auction_return_concentration": close_auction_return_concentration,
                "close_auction_volume_concentration": close_auction_volume_concentration,
                "morning_vs_afternoon_return_pct": morning_vs_afternoon_return_pct,
                "close_vwap_deviation_pct": close_vwap_deviation_pct,
                "amihud_illiquidity": amihud_illiquidity,
                "minute_bar_count": minute_bar_count,
                "is_complete_session": is_complete_session,
            }
        )

    return pd.DataFrame(daily_rows)

In [ ]:
reliance_minute_features = create_minute_daily_features(
    sample_minute_df,
    sample_minute_symbol,
)

print("Shape:", reliance_minute_features.shape)

reliance_minute_features.head()

In [ ]:
reliance_minute_features.info()

In [ ]:
duplicate_count = reliance_minute_features.duplicated(subset=["symbol", "pred_date"]).sum()

print("Duplicate symbol-date rows:", duplicate_count)

In [ ]:
reliance_minute_missingness = (reliance_minute_features.isna().sum().to_frame("missing_count"))
reliance_minute_missingness["missing_pct"] = (reliance_minute_missingness["missing_count"]/ len(reliance_minute_features)* 100)
reliance_minute_missingness

In [ ]:
reliance_minute_features.loc[~reliance_minute_features["is_complete_session"]].head(10)

In [ ]:
minute_feature_frames = []
minute_processing_errors = []

for index, file_path in enumerate(minute_files, start=1):
    symbol = file_path.stem

    try:
        stock_minute_df = pd.read_parquet(file_path)

        stock_minute_features = create_minute_daily_features(
            stock_minute_df,
            symbol,
        )

        minute_feature_frames.append(stock_minute_features)

    except Exception as error:
        minute_processing_errors.append(
            {
                "symbol": symbol,
                "file": file_path.name,
                "error": str(error),
            }
        )

    if index % 20 == 0 or index == len(minute_files):
        print(f"Processed {index}/{len(minute_files)} files")

In [ ]:
print("Files processed successfully:", len(minute_feature_frames))
print("Files with errors:", len(minute_processing_errors))

In [ ]:
minute_processing_errors_df = pd.DataFrame(minute_processing_errors)

minute_processing_errors_df

In [ ]:
minute_features_df = pd.concat(
    minute_feature_frames,
    ignore_index=True,
)

minute_features_df = minute_features_df.sort_values(
    ["symbol", "pred_date"]
).reset_index(drop=True)

print("Combined minute feature shape:", minute_features_df.shape)

minute_features_df.head()

In [ ]:
print("Symbols:", minute_features_df["symbol"].nunique())
print("Trading dates:", minute_features_df["pred_date"].nunique())

print("Date range:",minute_features_df["pred_date"].min(),"to",minute_features_df["pred_date"].max(),)

In [ ]:
minute_duplicate_count = minute_features_df.duplicated(
    subset=["symbol", "pred_date"]
).sum()

print("Duplicate symbol-date rows:", minute_duplicate_count)

In [ ]:
minute_feature_columns = [
    "intraday_realized_volatility_pct",
    "close_auction_return_concentration",
    "close_auction_volume_concentration",
    "morning_vs_afternoon_return_pct",
    "close_vwap_deviation_pct",
    "amihud_illiquidity",
]


minute_quality_columns = [
    "minute_bar_count",
    "is_complete_session",
]

In [ ]:
minute_feature_missingness = minute_features_df[
    minute_feature_columns + minute_quality_columns
].isna().sum().to_frame("missing_count")

minute_feature_missingness["missing_pct"] = (
    minute_feature_missingness["missing_count"]
    / len(minute_features_df)
    * 100
)

minute_feature_missingness.sort_values(
    "missing_pct",
    ascending=False,
)

In [ ]:
minute_features_df[minute_feature_columns] = (minute_features_df[minute_feature_columns].replace([np.inf, -np.inf], np.nan))

In [ ]:
minute_features_df["is_complete_session"].value_counts(dropna=False)

In [ ]:
minute_features_df["minute_bar_count"].describe()

In [ ]:
minute_features_df["minute_bar_count"].value_counts().sort_index().tail(10)

In [ ]:
assert minute_duplicate_count == 0

assert (minute_features_df["minute_bar_count"] > 0).all()

assert minute_features_df["is_complete_session"].isin([True, False]).all()

valid_volume_concentration = minute_features_df["close_auction_volume_concentration"].dropna()

assert valid_volume_concentration.between(0, 1).all()

print("Minute feature sanity checks passed.")

In [ ]:
minute_features_path = (PROCESSED_DATA_DIR / "minute_features_v1.parquet")

minute_features_df.to_parquet(minute_features_path,index=False,)

print("Saved minute feature panel to:")
print(minute_features_path)

### Merge Daily and Minute Feature Panels

In [ ]:
print(daily_features_df[["symbol", "pred_date"]].head())

print(minute_features_df[["symbol", "pred_date"]].head())

In [ ]:
print(daily_features_df[["symbol", "pred_date"]].dtypes)

print(minute_features_df[["symbol", "pred_date"]].dtypes)

In [ ]:
daily_keys = daily_features_df[["symbol", "pred_date"]].drop_duplicates()

minute_keys = minute_features_df[["symbol", "pred_date"]].drop_duplicates()

matching_keys = daily_keys.merge(minute_keys,on=["symbol", "pred_date"],how="inner",)

print("Daily stock-date keys:", len(daily_keys))
print("Minute stock-date keys:", len(minute_keys))
print("Matching stock-date keys:", len(matching_keys))
print("Daily coverage matched by minute data: {:.2f}%".format(len(matching_keys) / len(daily_keys) * 100))

In [ ]:
model_df = daily_features_df.merge(minute_features_df,on=["symbol", "pred_date"],how="left",validate="one_to_one",)

print("Merged model dataset shape:", model_df.shape)

model_df.head()

In [ ]:
print("Daily rows before merge:", len(daily_features_df))
print("Rows after merge:", len(model_df))

assert len(model_df) == len(daily_features_df)

print("All daily rows were preserved.")

In [ ]:
minute_merge_coverage = model_df[minute_feature_columns].notna().any(axis=1).mean() * 100

print("Rows with at least one minute feature: {:.2f}%".format(minute_merge_coverage))

In [ ]:
unmatched_minute_rows = model_df[model_df[minute_feature_columns].isna().all(axis=1)]

print("Rows without minute features:", len(unmatched_minute_rows))

unmatched_minute_rows[["pred_date", "symbol", "open", "close"]].head(10)

In [ ]:
unmatched_by_symbol = (unmatched_minute_rows.groupby("symbol").size().sort_values(ascending=False))

unmatched_by_symbol.head(20)

In [ ]:
model_duplicate_count = model_df.duplicated(subset=["symbol", "pred_date"]).sum()

print("Duplicate symbol-date rows after merge:", model_duplicate_count)

assert model_duplicate_count == 0

In [ ]:
all_feature_columns = daily_feature_columns + minute_feature_columns

print("Daily features:", len(daily_feature_columns))
print("Minute features:", len(minute_feature_columns))
print("Total Version 1 features so far:", len(all_feature_columns))

In [ ]:
combined_feature_missingness = model_df[all_feature_columns].isna().sum().to_frame("missing_count")

combined_feature_missingness["missing_pct"] = (combined_feature_missingness["missing_count"] / len(model_df) * 100)

combined_feature_missingness.sort_values("missing_pct", ascending=False)

In [ ]:
merged_panel_path = PROCESSED_DATA_DIR / "model_panel_daily_minute_v1.parquet"

model_df.to_parquet(merged_panel_path,index=False,)

print("Saved merged daily-minute panel to:")
print(merged_panel_path)

### Cross-Sectional Feature Engineering — Version 1

In [ ]:
cross_sectional_df = model_df.copy()

cross_sectional_df = cross_sectional_df.sort_values(["pred_date", "symbol"]).reset_index(drop=True)

print("Starting shape:", cross_sectional_df.shape)

In [ ]:
cross_sectional_df["return_1d_rank_pct"] = (cross_sectional_df.groupby("pred_date")["return_1d"].rank(method="average", pct=True))

In [ ]:
return_mean_by_date = (cross_sectional_df.groupby("pred_date")["return_1d"].transform("mean"))

return_std_by_date = (cross_sectional_df.groupby("pred_date")["return_1d"].transform("std"))

cross_sectional_df["return_1d_zscore"] = (cross_sectional_df["return_1d"] - return_mean_by_date) / return_std_by_date

In [ ]:
cross_sectional_df["universe_mean_return_1d"] = (cross_sectional_df.groupby("pred_date")["return_1d"].transform("mean"))

cross_sectional_df["cross_sectional_return_dispersion"] = (cross_sectional_df.groupby("pred_date")["return_1d"].transform("std"))

cross_sectional_df["market_breadth"] = (cross_sectional_df["return_1d"].gt(0).groupby(cross_sectional_df["pred_date"]).transform("mean"))

cross_sectional_df["aggregate_universe_volatility"] = (cross_sectional_df.groupby("pred_date")["daily_volatility_20d"].transform("median"))

In [ ]:
universe_return_df = (cross_sectional_df[["pred_date", "universe_mean_return_1d"]].drop_duplicates(subset=["pred_date"]).sort_values("pred_date").reset_index(drop=True))
universe_return_df.head()

In [ ]:
cross_sectional_df["stock_market_product"] = (cross_sectional_df["return_1d"] * cross_sectional_df["universe_mean_return_1d"])

cross_sectional_df["market_return_squared"] = (cross_sectional_df["universe_mean_return_1d"] ** 2)

In [ ]:
cross_sectional_df["rolling_mean_stock_return_60d"] = (cross_sectional_df.groupby("symbol")["return_1d"].transform(
        lambda series: series.rolling(
            window=60,
            min_periods=40,
        ).mean()
    )
)

cross_sectional_df["rolling_mean_market_return_60d"] = (
    cross_sectional_df.groupby("symbol")["universe_mean_return_1d"]
    .transform(
        lambda series: series.rolling(
            window=60,
            min_periods=40,
        ).mean()
    )
)

cross_sectional_df["rolling_mean_stock_market_product_60d"] = (
    cross_sectional_df.groupby("symbol")["stock_market_product"]
    .transform(
        lambda series: series.rolling(
            window=60,
            min_periods=40,
        ).mean()
    )
)

In [ ]:
cross_sectional_df["rolling_mean_market_squared_60d"] = (
    cross_sectional_df.groupby("symbol")["market_return_squared"]
    .transform(
        lambda series: series.rolling(
            window=60,
            min_periods=40,
        ).mean()
    )
)

cross_sectional_df["rolling_market_variance_60d"] = (
    cross_sectional_df["rolling_mean_market_squared_60d"]
    - cross_sectional_df["rolling_mean_market_return_60d"] ** 2
)

In [ ]:
cross_sectional_df["rolling_stock_market_covariance_60d"] = (
    cross_sectional_df["rolling_mean_stock_market_product_60d"]
    - (
        cross_sectional_df["rolling_mean_stock_return_60d"]
        * cross_sectional_df["rolling_mean_market_return_60d"]
    )
)

cross_sectional_df["rolling_beta_to_universe_60d"] = (
    cross_sectional_df["rolling_stock_market_covariance_60d"]
    / cross_sectional_df["rolling_market_variance_60d"]
)

In [ ]:
cross_sectional_feature_columns = ["return_1d_rank_pct","return_1d_zscore","rolling_beta_to_universe_60d","cross_sectional_return_dispersion","market_breadth","aggregate_universe_volatility",]
print("Cross-sectional features:", len(cross_sectional_feature_columns))

In [ ]:
cross_sectional_df[cross_sectional_feature_columns] = (cross_sectional_df[cross_sectional_feature_columns].replace([np.inf, -np.inf], np.nan))

In [ ]:
cross_sectional_missingness = (
    cross_sectional_df[cross_sectional_feature_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

cross_sectional_missingness["missing_pct"] = (
    cross_sectional_missingness["missing_count"]
    / len(cross_sectional_df)
    * 100
)

cross_sectional_missingness.sort_values(
    "missing_pct",
    ascending=False,
)

In [ ]:
valid_return_ranks = cross_sectional_df["return_1d_rank_pct"].dropna()
valid_market_breadth = cross_sectional_df["market_breadth"].dropna()
valid_dispersion = cross_sectional_df["cross_sectional_return_dispersion"].dropna()
valid_aggregate_volatility = cross_sectional_df["aggregate_universe_volatility"].dropna()

assert valid_return_ranks.between(0, 1).all()
assert valid_market_breadth.between(0, 1).all()
assert (valid_dispersion >= 0).all()
assert (valid_aggregate_volatility >= 0).all()

print("Cross-sectional feature sanity checks passed.")

In [ ]:
sample_cross_sectional_date = (
    cross_sectional_df["pred_date"]
    .dropna()
    .sort_values()
    .iloc[-100]
)

cross_sectional_df.loc[
    cross_sectional_df["pred_date"] == sample_cross_sectional_date,
    [
        "pred_date",
        "symbol",
        "return_1d",
        "return_1d_rank_pct",
        "return_1d_zscore",
        "universe_mean_return_1d",
        "cross_sectional_return_dispersion",
        "market_breadth",
        "aggregate_universe_volatility",
        "rolling_beta_to_universe_60d",
    ],
].sort_values("return_1d", ascending=False).head(10)

In [ ]:
all_feature_columns = (
    daily_feature_columns
    + minute_feature_columns
    + cross_sectional_feature_columns
)

print("Daily features:", len(daily_feature_columns))
print("Minute features:", len(minute_feature_columns))
print("Cross-sectional features:", len(cross_sectional_feature_columns))
print("Total Version 1 features:", len(all_feature_columns))

In [ ]:
temporary_cross_sectional_columns = [
    "stock_market_product",
    "market_return_squared",
    "rolling_mean_stock_return_60d",
    "rolling_mean_market_return_60d",
    "rolling_mean_stock_market_product_60d",
    "rolling_mean_market_squared_60d",
    "rolling_market_variance_60d",
    "rolling_stock_market_covariance_60d",
]

cross_sectional_df = cross_sectional_df.drop(
    columns=temporary_cross_sectional_columns
)

In [ ]:
model_features_df = cross_sectional_df.copy()

print("Completed feature panel shape:", model_features_df.shape)

In [ ]:
complete_feature_rows = model_features_df.dropna(
    subset=all_feature_columns
)

print("Total rows:", len(model_features_df))
print("Rows with all Version 1 features:", len(complete_feature_rows))
print(
    "Complete-feature coverage: {:.2f}%".format(
        len(complete_feature_rows)
        / len(model_features_df)
        * 100
    )
)

In [ ]:
full_feature_panel_path = (
    PROCESSED_DATA_DIR
    / "model_features_full_v1.parquet"
)

model_features_df.to_parquet(
    full_feature_panel_path,
    index=False,
)

print("Saved full Version 1 feature panel to:")
print(full_feature_panel_path)

### Chronological Train, Validation and Test Splits

In [ ]:
split_df = model_features_df.copy()

split_df = split_df.sort_values(["pred_date", "symbol"]).reset_index(drop=True)

print("Starting rows:", len(split_df))
print("Date range:", split_df["pred_date"].min(), "to", split_df["pred_date"].max())

In [ ]:
VALID_PERIOD_START = pd.Timestamp("2024-04-01")
TEST_PERIOD_START = pd.Timestamp("2025-05-01")
EMBARGO_SESSIONS = 5

print("Validation period begins around:", VALID_PERIOD_START.date())
print("Test period begins around:", TEST_PERIOD_START.date())
print("Embargo sessions:", EMBARGO_SESSIONS)

In [ ]:
available_pred_dates = pd.Index(split_df["pred_date"].dropna().sort_values().unique())

print("Number of prediction dates:", len(available_pred_dates))
print("First date:", available_pred_dates.min())
print("Last date:", available_pred_dates.max())

In [ ]:
train_valid_embargo_dates = available_pred_dates[available_pred_dates >= VALID_PERIOD_START][:EMBARGO_SESSIONS]

print("Train-validation embargo dates:")

for date in train_valid_embargo_dates:
    print(pd.Timestamp(date).date())

In [ ]:
actual_valid_start = available_pred_dates[available_pred_dates > train_valid_embargo_dates[-1]][0]

print("Actual validation start:", pd.Timestamp(actual_valid_start).date())

In [ ]:
valid_test_embargo_dates = available_pred_dates[available_pred_dates >= TEST_PERIOD_START][:EMBARGO_SESSIONS]

print("Validation-test embargo dates:")

for date in valid_test_embargo_dates:
    print(pd.Timestamp(date).date())

In [ ]:
actual_test_start = available_pred_dates[available_pred_dates > valid_test_embargo_dates[-1]][0]

print("Actual test start:", pd.Timestamp(actual_test_start).date())

In [ ]:
split_df["split"] = "unused"

train_mask = split_df["pred_date"] < VALID_PERIOD_START

train_valid_embargo_mask = split_df["pred_date"].isin(train_valid_embargo_dates)

valid_mask = ((split_df["pred_date"] >= actual_valid_start)& (split_df["pred_date"] < TEST_PERIOD_START))

valid_test_embargo_mask = split_df["pred_date"].isin(valid_test_embargo_dates)

test_mask = split_df["pred_date"] >= actual_test_start

split_df.loc[train_mask, "split"] = "train"
split_df.loc[train_valid_embargo_mask, "split"] = "embargo"
split_df.loc[valid_mask, "split"] = "valid"
split_df.loc[valid_test_embargo_mask, "split"] = "embargo"
split_df.loc[test_mask, "split"] = "test"

split_df["split"].value_counts()

In [ ]:
split_date_summary = split_df.groupby("split").agg(
    start_date=("pred_date", "min"),
    end_date=("pred_date", "max"),
    rows=("pred_date", "size"),
    dates=("pred_date", "nunique"),
    symbols=("symbol", "nunique"),
)

split_date_summary

In [ ]:
unused_rows = split_df.loc[split_df["split"] == "unused",["pred_date", "symbol"]]

print("Unused rows:", len(unused_rows))

unused_rows.head()

In [ ]:
train_end = split_df.loc[split_df["split"] == "train","pred_date"].max()

valid_start = split_df.loc[split_df["split"] == "valid","pred_date"].min()

valid_end = split_df.loc[split_df["split"] == "valid","pred_date"].max()

test_start = split_df.loc[split_df["split"] == "test","pred_date"].min()

print("Train end:", train_end)
print("Validation start:", valid_start)
print("Validation end:", valid_end)
print("Test start:", test_start)

assert train_end < valid_start
assert valid_end < test_start

print("Chronological ordering confirmed.")

In [ ]:
print("Train-validation embargo sessions:",split_df.loc[split_df["pred_date"].isin(train_valid_embargo_dates),"pred_date"].nunique())

print("Validation-test embargo sessions:",split_df.loc[split_df["pred_date"].isin(valid_test_embargo_dates),"pred_date"].nunique())

assert len(train_valid_embargo_dates) == EMBARGO_SESSIONS
assert len(valid_test_embargo_dates) == EMBARGO_SESSIONS

print("Embargo checks passed.")


In [ ]:
model_split_df = split_df.copy()

print("Completed split panel shape:", model_split_df.shape)

In [ ]:
feature_coverage_by_split = (model_split_df.groupby("split")[all_feature_columns].apply(lambda frame: frame.notna().mean() * 100))

feature_coverage_by_split.T

In [ ]:
complete_row_summary = []

for split_name in ["train", "valid", "test"]:
    split_subset = model_split_df.loc[
        model_split_df["split"] == split_name
    ]

    complete_rows = split_subset[
        all_feature_columns
    ].notna().all(axis=1).sum()

    complete_row_summary.append(
        {
            "split": split_name,
            "total_rows": len(split_subset),
            "complete_rows": complete_rows,
            "complete_pct": complete_rows / len(split_subset) * 100,
        }
    )

complete_row_summary_df = pd.DataFrame(complete_row_summary)

complete_row_summary_df

In [ ]:
split_panel_path = (PROCESSED_DATA_DIR/ "model_features_with_splits_v1.parquet")

model_split_df.to_parquet(split_panel_path,index=False,)

print("Saved split feature panel to:")
print(split_panel_path)

### ML Dataset Preparation

Prepare the final modelling datasets.

Steps:

1. Define metadata columns
2. Define target columns
3. Define feature columns
4. Remove leakage columns
5. Drop warm-up rows
6. Create train / validation / test datasets
7. Create X and y matrices

In [ ]:
METADATA_COLUMNS = ["symbol","pred_date","target_date","split",]

In [ ]:
TARGET_COLUMNS = [
    "actual_return_pct",
    "actual_direction",
    "actual_magnitude_pct",
]

In [ ]:
LEAKAGE_COLUMNS = [
    "next_open",
    "actual_return_pct",
    "actual_direction",
    "actual_magnitude_pct",
    "target_date",
    "split",
]

In [ ]:
FEATURE_COLUMNS = sorted(list(set(all_feature_columns)| {"symbol"}))

print("Number of model features:", len(FEATURE_COLUMNS))

FEATURE_COLUMNS[:10]

In [ ]:
leaking_features = sorted(
    set(FEATURE_COLUMNS)
    & set(LEAKAGE_COLUMNS)
)

assert len(leaking_features) == 0

print("No leakage columns found in feature list.")

In [ ]:
ml_df = model_split_df.copy()

print("Rows:", len(ml_df))
print("Columns:", len(ml_df.columns))

In [ ]:
initial_rows = len(ml_df)

ml_df = ml_df.dropna(subset=daily_feature_columns).reset_index(drop=True)

print("Dropped rows:", initial_rows - len(ml_df))
print("Remaining rows:", len(ml_df))

In [ ]:
ml_df["symbol"] = (ml_df["symbol"].astype("category"))

ml_df["symbol"].dtype

In [ ]:
train_df = ml_df.loc[ml_df["split"] == "train"].copy()

valid_df = ml_df.loc[ml_df["split"] == "valid"].copy()

test_df = ml_df.loc[ml_df["split"] == "test"].copy()

print(len(train_df), len(valid_df), len(test_df))

In [ ]:
train_metadata = train_df[METADATA_COLUMNS].reset_index(drop=True)

valid_metadata = valid_df[METADATA_COLUMNS].reset_index(drop=True)

test_metadata = test_df[METADATA_COLUMNS].reset_index(drop=True)

In [ ]:
X_train = train_df[FEATURE_COLUMNS].copy()

X_valid = valid_df[FEATURE_COLUMNS].copy()

X_test = test_df[FEATURE_COLUMNS].copy()

print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)

In [ ]:
y_direction_train = train_df["actual_direction"]

y_direction_valid = valid_df["actual_direction"]

y_direction_test = test_df["actual_direction"]

In [ ]:
y_magnitude_train = np.log1p(train_df["actual_magnitude_pct"])

y_magnitude_valid = np.log1p(valid_df["actual_magnitude_pct"])

y_magnitude_test = np.log1p(test_df["actual_magnitude_pct"])

In [ ]:
print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Test rows:", len(X_test))

assert len(X_train) == len(y_direction_train)
assert len(X_valid) == len(y_direction_valid)
assert len(X_test) == len(y_direction_test)

assert len(X_train) == len(y_magnitude_train)

print("ML dataset preparation complete.")

### Baseline Direction Model

Train the first pooled LightGBM classifier to predict the direction of the next overnight return.

Target:
- actual_direction

Output:
- predicted direction
- predicted probability

In [ ]:
import lightgbm as lgb

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

In [ ]:
y_train_binary = (y_direction_train == 1).astype(int)

y_valid_binary = (y_direction_valid == 1).astype(int)

y_test_binary = (y_direction_test == 1).astype(int)

In [ ]:
categorical_features = [
    "symbol",
]

In [ ]:
train_dataset = lgb.Dataset(
    X_train,
    label=y_train_binary,
    categorical_feature=categorical_features,
    free_raw_data=False,
)

valid_dataset = lgb.Dataset(
    X_valid,
    label=y_valid_binary,
    categorical_feature=categorical_features,
    free_raw_data=False,
)

In [ ]:
direction_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
    "verbosity": -1,
}

In [ ]:
direction_model = lgb.train(
    params=direction_params,
    train_set=train_dataset,
    valid_sets=[train_dataset, valid_dataset],
    valid_names=["train", "valid"],
    num_boost_round=500,
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50),
    ],
)

In [ ]:
valid_probability = direction_model.predict(X_valid)

test_probability = direction_model.predict(X_test)

In [ ]:
valid_prediction = (valid_probability >= 0.5).astype(int)

test_prediction = (test_probability >= 0.5).astype(int)

In [ ]:
valid_prediction = np.where(valid_prediction == 1,1,-1,)

test_prediction = np.where(test_prediction == 1,1,-1,)

In [ ]:
validation_accuracy = accuracy_score(y_direction_valid,valid_prediction,)

print(f"Validation Accuracy: {validation_accuracy:.4f}")

In [ ]:
confusion_matrix(y_direction_valid,valid_prediction,)

In [ ]:
print(classification_report(y_direction_valid,valid_prediction,))

In [ ]:
direction_feature_importance = (pd.DataFrame(
        {
            "feature": direction_model.feature_name(),
            "importance": direction_model.feature_importance(
                importance_type="gain"
            ),
        }
    )
    .sort_values(
        "importance",
        ascending=False,
    )
)

direction_feature_importance.head(20)

In [ ]:
# print("="*80)
# print("VALIDATION ACCURACY")
# print("="*80)
# print(validation_accuracy)

# print("\n")
# print("="*80)
# print("CONFUSION MATRIX")
# print("="*80)
# print(confusion_matrix(y_direction_valid, valid_prediction))

# print("\n")
# print("="*80)
# print("CLASSIFICATION REPORT")
# print("="*80)
# print(classification_report(y_direction_valid, valid_prediction, digits=4))

# print("\n")
# print("="*80)
# print("TOP 20 FEATURES")
# print("="*80)
# print(direction_feature_importance.head(20).to_string(index=False))

# print("\n")
# print("="*80)
# print("PROBABILITY SUMMARY")
# print("="*80)
# print(pd.Series(valid_probability).describe())

#### Direction Feature Engineering — Version 3

Version 3 adds two feature families:

1. Daily and cross-sectional features derived from the existing daily panel.
2. Minute-derived daily features computed from raw one-minute bars.

The minute features are processed once and saved separately so that model experimentation
does not require rereading all 208 minute parquet files.

In [ ]:
V3_MINUTE_FEATURES_PATH = (
    PROCESSED_DATA_DIR / "direction_v3_minute_features.parquet"
)

V3_DAILY_FEATURES_PATH = (
    PROCESSED_DATA_DIR / "direction_v3_daily_features.parquet"
)

V3_MODEL_PANEL_PATH = (
    PROCESSED_DATA_DIR / "direction_v3_model_panel.parquet"
)

REBUILD_V3_MINUTE_FEATURES = False

print("Minute feature cache:", V3_MINUTE_FEATURES_PATH)
print("Daily feature panel:", V3_DAILY_FEATURES_PATH)
print("Final V3 model panel:", V3_MODEL_PANEL_PATH)
print("Rebuild minute features:", REBUILD_V3_MINUTE_FEATURES)

#### V3 Daily and Cross-Sectional Features

These features are inexpensive to calculate and therefore may be regenerated whenever
the V3 dataset is rebuilt.

All rolling statistics use current or historical information available by the close of
prediction date T. Historical overnight-return features are shifted before rolling because
the current row's overnight target is realised only at the next session's open.

In [ ]:
direction_v3_daily_df = daily_features_df.copy()

direction_v3_daily_df = (
    direction_v3_daily_df
    .sort_values(["symbol", "pred_date"])
    .reset_index(drop=True)
)

print("Starting V3 daily shape:", direction_v3_daily_df.shape)
print("Symbols:", direction_v3_daily_df["symbol"].nunique())
print(
    "Date range:",
    direction_v3_daily_df["pred_date"].min(),
    "to",
    direction_v3_daily_df["pred_date"].max(),
)

In [ ]:
existing_market_columns = [
    column
    for column in [
        "market_breadth",
        "cross_sectional_return_dispersion",
        "aggregate_universe_volatility",
    ]
    if column in direction_v3_daily_df.columns
]

print("Market columns already present:", existing_market_columns)

In [ ]:
market_daily_features = (
    direction_v3_daily_df
    .groupby("pred_date", as_index=False)
    .agg(
        universe_return_1d=(
            "return_1d",
            "mean",
        ),
        market_breadth=(
            "return_1d",
            lambda series: series.gt(0).mean(),
        ),
        cross_sectional_return_dispersion=(
            "return_1d",
            "std",
        ),
        aggregate_universe_volatility=(
            "daily_volatility_20d",
            "mean",
        ),
    )
    .sort_values("pred_date")
    .reset_index(drop=True)
)

market_daily_features.head()

In [ ]:
direction_v3_daily_df = direction_v3_daily_df.merge(
    market_daily_features,
    on="pred_date",
    how="left",
    validate="many_to_one",
)

print("Shape after market-feature merge:", direction_v3_daily_df.shape)

direction_v3_daily_df[
    [
        "pred_date",
        "symbol",
        "return_1d",
        "universe_return_1d",
        "market_breadth",
        "cross_sectional_return_dispersion",
        "aggregate_universe_volatility",
    ]
].head()

In [ ]:
assert direction_v3_daily_df[
    "market_breadth"
].dropna().between(0, 1).all()

assert (
    direction_v3_daily_df[
        "cross_sectional_return_dispersion"
    ].dropna() >= 0
).all()

assert (
    direction_v3_daily_df[
        "aggregate_universe_volatility"
    ].dropna() >= 0
).all()

market_feature_columns = [
    "universe_return_1d",
    "market_breadth",
    "cross_sectional_return_dispersion",
    "aggregate_universe_volatility",
]

print(
    direction_v3_daily_df[
        market_feature_columns
    ].isna().sum()
)

print("Market feature checks passed.")

#### V3 Cross-Sectional and Regime Features

The following features measure whether a stock's current movement is unusual relative
to the rest of the universe and whether the current market environment differs from its
recent history.

All rolling regime statistics use only the current and previous prediction dates.

In [ ]:
direction_v3_daily_df["idiosyncratic_return_1d"] = (
    direction_v3_daily_df["return_1d"]
    - direction_v3_daily_df["universe_return_1d"]
)

In [ ]:
direction_v3_daily_df["relative_return_rank_1d"] = (
    direction_v3_daily_df
    .groupby("pred_date")["return_1d"]
    .rank(
        method="average",
        pct=True,
    )
)

In [ ]:
direction_v3_daily_df["relative_momentum_rank_5d"] = (
    direction_v3_daily_df
    .groupby("pred_date")["return_5d"]
    .rank(
        method="average",
        pct=True,
    )
)

In [ ]:
market_regime_df = (
    market_daily_features[
        [
            "pred_date",
            "market_breadth",
            "cross_sectional_return_dispersion",
        ]
    ]
    .sort_values("pred_date")
    .reset_index(drop=True)
)

# ---------------- Breadth regime ---------------- #

market_regime_df["breadth_mean_20d_lagged"] = (
    market_regime_df["market_breadth"]
    .shift(1)
    .rolling(window=20, min_periods=20)
    .mean()
)

market_regime_df["breadth_std_20d_lagged"] = (
    market_regime_df["market_breadth"]
    .shift(1)
    .rolling(window=20, min_periods=20)
    .std()
)

breadth_std = (
    market_regime_df["breadth_std_20d_lagged"]
    .replace(0, np.nan)
)

market_regime_df["breadth_shock_zscore"] = (
    market_regime_df["market_breadth"]
    - market_regime_df["breadth_mean_20d_lagged"]
) / breadth_std

# ---------------- Dispersion regime ---------------- #

market_regime_df["dispersion_mean_20d_lagged"] = (
    market_regime_df["cross_sectional_return_dispersion"]
    .shift(1)
    .rolling(window=20, min_periods=20)
    .mean()
)

market_regime_df["dispersion_std_20d_lagged"] = (
    market_regime_df["cross_sectional_return_dispersion"]
    .shift(1)
    .rolling(window=20, min_periods=20)
    .std()
)

dispersion_std = (
    market_regime_df["dispersion_std_20d_lagged"]
    .replace(0, np.nan)
)

market_regime_df["dispersion_regime_zscore"] = (
    market_regime_df["cross_sectional_return_dispersion"]
    - market_regime_df["dispersion_mean_20d_lagged"]
) / dispersion_std

market_regime_df.head()

In [ ]:
direction_v3_daily_df = direction_v3_daily_df.merge(
    market_regime_df[
        [
            "pred_date",
            "breadth_shock_zscore",
            "dispersion_regime_zscore",
        ]
    ],
    on="pred_date",
    how="left",
    validate="many_to_one",
)

print("Shape after regime-feature merge:", direction_v3_daily_df.shape)

In [ ]:
v3_cross_sectional_feature_columns = [
    "idiosyncratic_return_1d",
    "relative_return_rank_1d",
    "relative_momentum_rank_5d",
    "breadth_shock_zscore",
    "dispersion_regime_zscore",
]

assert direction_v3_daily_df[
    "relative_return_rank_1d"
].dropna().between(0, 1).all()

assert direction_v3_daily_df[
    "relative_momentum_rank_5d"
].dropna().between(0, 1).all()

assert not np.isinf(
    direction_v3_daily_df[
        v3_cross_sectional_feature_columns
    ].to_numpy(dtype=float)
).any()

direction_v3_daily_df[
    v3_cross_sectional_feature_columns
].describe().T

In [ ]:
direction_v3_daily_df[
    v3_cross_sectional_feature_columns
].isna().sum()

In [ ]:
v3_na_diagnostics = pd.DataFrame({
    "nan_count": direction_v3_daily_df[
        v3_cross_sectional_feature_columns
    ].isna().sum(),
    "nan_pct": (
        direction_v3_daily_df[
            v3_cross_sectional_feature_columns
        ].isna().mean() * 100
    ),
})

v3_na_diagnostics

In [ ]:
for column in [
    "breadth_shock_zscore",
    "dispersion_regime_zscore",
]:
    print(f"\n{column}")
    print(
        direction_v3_daily_df.loc[
            direction_v3_daily_df[column].isna(),
            "pred_date",
        ].drop_duplicates().sort_values().head(25).tolist()
    )

#### V3 Gap Behaviour Features

These features describe whether the current overnight gap is unusually large relative
to the stock's own history and whether the stock reverses or continues that gap during
the trading session.

Only information available by the close of prediction date T is used.

In [ ]:
required_gap_columns = [
    "symbol",
    "pred_date",
    "open",
    "close",
]

missing_gap_columns = [
    column
    for column in required_gap_columns
    if column not in direction_v3_daily_df.columns
]

assert not missing_gap_columns, (
    f"Missing required columns: {missing_gap_columns}"
)

direction_v3_daily_df = (
    direction_v3_daily_df
    .sort_values(["symbol", "pred_date"])
    .reset_index(drop=True)
)

In [ ]:
direction_v3_daily_df["current_gap_pct"] = (
    direction_v3_daily_df["open"]
    / direction_v3_daily_df.groupby("symbol")["close"].shift(1)
    - 1
) * 100

direction_v3_daily_df["intraday_return_pct"] = (
    direction_v3_daily_df["close"]
    / direction_v3_daily_df["open"]
    - 1
) * 100

In [ ]:
symbol_groups = direction_v3_daily_df.groupby(
    "symbol",
    group_keys=False,
)

direction_v3_daily_df["gap_mean_20d_lagged"] = (
    symbol_groups["current_gap_pct"]
    .transform(
        lambda series: (
            series.shift(1)
            .rolling(window=20, min_periods=20)
            .mean()
        )
    )
)

direction_v3_daily_df["gap_std_20d_lagged"] = (
    symbol_groups["current_gap_pct"]
    .transform(
        lambda series: (
            series.shift(1)
            .rolling(window=20, min_periods=20)
            .std()
        )
    )
)

In [ ]:
gap_std_safe = (
    direction_v3_daily_df["gap_std_20d_lagged"]
    .replace(0, np.nan)
)

direction_v3_daily_df["standardized_current_gap"] = (
    direction_v3_daily_df["current_gap_pct"]
    - direction_v3_daily_df["gap_mean_20d_lagged"]
) / gap_std_safe

direction_v3_daily_df["gap_intraday_reversal"] = (
    -np.sign(direction_v3_daily_df["current_gap_pct"])
    * direction_v3_daily_df["intraday_return_pct"]
)

In [ ]:
v3_gap_feature_columns = [
    "standardized_current_gap",
    "gap_intraday_reversal",
]

assert not np.isinf(
    direction_v3_daily_df[
        v3_gap_feature_columns
    ].to_numpy(dtype=float)
).any()

print(
    direction_v3_daily_df[
        v3_gap_feature_columns
    ].isna().sum()
)

direction_v3_daily_df[
    [
        "symbol",
        "pred_date",
        "current_gap_pct",
        "intraday_return_pct",
        "standardized_current_gap",
        "gap_intraday_reversal",
    ]
].describe().T

In [ ]:
direction_v3_daily_df[
    v3_gap_feature_columns
].isna().sum()

In [ ]:
direction_v3_daily_df.to_parquet(
    V3_DAILY_FEATURES_PATH,
    index=False,
)

print("Saved:", V3_DAILY_FEATURES_PATH)
print(direction_v3_daily_df.shape)

#### V3 Minute-Derived Daily Features

The raw one-minute files are processed once to create one row per symbol and trading date.

The features capture:

- Intraday price path
- Realised volatility and asymmetry
- Early-versus-late session behaviour
- Volume concentration and closing pressure
- Closing-session trend
- Data coverage and session completeness

The resulting minute feature dataset is saved separately before being merged with the
V3 daily feature panel.

In [ ]:
V3_MINUTE_FEATURE_COLUMNS = [
    "intraday_path_efficiency",
    "intraday_realized_volatility",
    "intraday_realized_skewness",
    "intraday_up_minute_fraction",
    "late_session_volatility_share_60m",
    "early_session_volatility_share_60m",
    "largest_minute_move_share",
    "signed_closing_volume_pressure",
    "closing_volume_share_60m",
    "closing_return_60m",
    "closing_trend_slope",
    "intraday_high_low_range_pct",
    "close_location_in_range",
    "volume_concentration_hhi",
    "minute_bar_coverage",
]

In [ ]:
def create_direction_v3_minute_features(
    minute_df: pd.DataFrame,
    symbol: str,
    expected_bars_per_session: int = 375,
) -> pd.DataFrame:
    """
    Convert one symbol's raw one-minute bars into one row per trading session.

    All features use information available up to the close of session T.
    """

    required_columns = {
        "timestamp",
        "open",
        "high",
        "low",
        "close",
        "volume",
    }

    missing_columns = required_columns.difference(minute_df.columns)

    if missing_columns:
        raise ValueError(
            f"{symbol}: missing minute columns: {sorted(missing_columns)}"
        )

    df = minute_df[
        [
            "timestamp",
            "open",
            "high",
            "low",
            "close",
            "volume",
        ]
    ].copy()

    df["timestamp"] = pd.to_datetime(df["timestamp"])

    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    df = (
        df.dropna(
            subset=[
                "timestamp",
                "open",
                "high",
                "low",
                "close",
            ]
        )
        .sort_values("timestamp")
        .drop_duplicates("timestamp", keep="last")
        .reset_index(drop=True)
    )

    df = df[
        (df["open"] > 0)
        & (df["high"] > 0)
        & (df["low"] > 0)
        & (df["close"] > 0)
    ].copy()

    df["volume"] = df["volume"].fillna(0).clip(lower=0)

    df["pred_date"] = df["timestamp"].dt.normalize()

    # Remove bars outside the regular session.
    minute_of_day = (
        df["timestamp"].dt.hour * 60
        + df["timestamp"].dt.minute
    )

    session_start = 9 * 60 + 15
    session_end = 15 * 60 + 29

    df = df[
        minute_of_day.between(
            session_start,
            session_end,
        )
    ].copy()

    if df.empty:
        return pd.DataFrame(
            columns=[
                "symbol",
                "pred_date",
                *V3_MINUTE_FEATURE_COLUMNS,
            ]
        )

    output_rows = []

    for pred_date, day in df.groupby(
        "pred_date",
        sort=True,
    ):
        day = (
            day.sort_values("timestamp")
            .reset_index(drop=True)
        )

        n_bars = len(day)

        if n_bars < 2:
            continue

        close = day["close"].to_numpy(dtype=float)
        open_price = day["open"].to_numpy(dtype=float)
        high = day["high"].to_numpy(dtype=float)
        low = day["low"].to_numpy(dtype=float)
        volume = day["volume"].to_numpy(dtype=float)

        # Log returns are additive and stable for path calculations.
        minute_log_returns = np.diff(np.log(close))

        finite_returns = minute_log_returns[
            np.isfinite(minute_log_returns)
        ]

        if finite_returns.size == 0:
            continue

        absolute_returns = np.abs(finite_returns)
        squared_returns = finite_returns ** 2

        total_absolute_path = absolute_returns.sum()

        session_log_move = abs(
            np.log(close[-1] / open_price[0])
        )

        if total_absolute_path > 0:
            intraday_path_efficiency = (
                session_log_move / total_absolute_path
            )
        else:
            intraday_path_efficiency = 0.0

        # Numerical and bar-construction differences can otherwise
        # produce tiny values above one.
        intraday_path_efficiency = float(
            np.clip(
                intraday_path_efficiency,
                0.0,
                1.0,
            )
        )

        intraday_realized_volatility = float(
            np.sqrt(squared_returns.sum()) * 100
        )

        if (
            finite_returns.size >= 3
            and np.std(finite_returns, ddof=1) > 0
        ):
            intraday_realized_skewness = float(
                pd.Series(finite_returns).skew()
            )
        else:
            intraday_realized_skewness = 0.0

        intraday_up_minute_fraction = float(
            np.mean(finite_returns > 0)
        )

        return_count = len(finite_returns)
        window_size = min(60, return_count)

        total_squared_return = squared_returns.sum()

        if total_squared_return > 0:
            early_session_volatility_share_60m = float(
                squared_returns[:window_size].sum()
                / total_squared_return
            )

            late_session_volatility_share_60m = float(
                squared_returns[-window_size:].sum()
                / total_squared_return
            )

            largest_minute_move_share = float(
                squared_returns.max()
                / total_squared_return
            )
        else:
            early_session_volatility_share_60m = 0.0
            late_session_volatility_share_60m = 0.0
            largest_minute_move_share = 0.0

        total_volume = volume.sum()
        closing_bar_count = min(60, n_bars)

        closing_volume = volume[-closing_bar_count:]
        closing_close = close[-closing_bar_count:]

        if total_volume > 0:
            closing_volume_share_60m = float(
                closing_volume.sum() / total_volume
            )

            volume_weights = volume / total_volume

            volume_concentration_hhi = float(
                np.square(volume_weights).sum()
            )
        else:
            closing_volume_share_60m = 0.0
            volume_concentration_hhi = 0.0

        # Assign each closing bar's volume the sign of its close-to-close move.
        closing_price_changes = np.diff(
            close[-(closing_bar_count + 1):]
        )

        if closing_price_changes.size == closing_volume.size:
            closing_signs = np.sign(closing_price_changes)
            signed_closing_volume = (
                closing_signs * closing_volume
            ).sum()
        else:
            closing_signs = np.sign(
                np.diff(closing_close, prepend=closing_close[0])
            )

            signed_closing_volume = (
                closing_signs * closing_volume
            ).sum()

        if closing_volume.sum() > 0:
            signed_closing_volume_pressure = float(
                signed_closing_volume
                / closing_volume.sum()
            )
        else:
            signed_closing_volume_pressure = 0.0

        if n_bars > closing_bar_count:
            closing_reference_price = close[
                -(closing_bar_count + 1)
            ]
        else:
            closing_reference_price = open_price[0]

        closing_return_60m = float(
            (
                close[-1]
                / closing_reference_price
                - 1
            )
            * 100
        )

        if closing_bar_count >= 2:
            closing_x = np.arange(
                closing_bar_count,
                dtype=float,
            )

            closing_log_prices = np.log(closing_close)

            closing_trend_slope = float(
                np.polyfit(
                    closing_x,
                    closing_log_prices,
                    1,
                )[0]
                * 100
            )
        else:
            closing_trend_slope = 0.0

        session_high = np.max(high)
        session_low = np.min(low)

        intraday_high_low_range_pct = float(
            (
                session_high
                / session_low
                - 1
            )
            * 100
        )

        price_range = session_high - session_low

        if price_range > 0:
            close_location_in_range = float(
                (
                    close[-1] - session_low
                )
                / price_range
            )
        else:
            close_location_in_range = 0.5

        minute_bar_coverage = float(
            min(
                n_bars / expected_bars_per_session,
                1.0,
            )
        )

        output_rows.append(
            {
                "symbol": symbol,
                "pred_date": pd.Timestamp(pred_date),
                "intraday_path_efficiency": intraday_path_efficiency,
                "intraday_realized_volatility": intraday_realized_volatility,
                "intraday_realized_skewness": intraday_realized_skewness,
                "intraday_up_minute_fraction": intraday_up_minute_fraction,
                "late_session_volatility_share_60m": (
                    late_session_volatility_share_60m
                ),
                "early_session_volatility_share_60m": (
                    early_session_volatility_share_60m
                ),
                "largest_minute_move_share": largest_minute_move_share,
                "signed_closing_volume_pressure": (
                    signed_closing_volume_pressure
                ),
                "closing_volume_share_60m": closing_volume_share_60m,
                "closing_return_60m": closing_return_60m,
                "closing_trend_slope": closing_trend_slope,
                "intraday_high_low_range_pct": (
                    intraday_high_low_range_pct
                ),
                "close_location_in_range": close_location_in_range,
                "volume_concentration_hhi": volume_concentration_hhi,
                "minute_bar_coverage": minute_bar_coverage,
            }
        )

    result = pd.DataFrame(output_rows)

    if result.empty:
        return pd.DataFrame(
            columns=[
                "symbol",
                "pred_date",
                *V3_MINUTE_FEATURE_COLUMNS,
            ]
        )

    result = (
        result[
            [
                "symbol",
                "pred_date",
                *V3_MINUTE_FEATURE_COLUMNS,
            ]
        ]
        .sort_values(["symbol", "pred_date"])
        .reset_index(drop=True)
    )

    return result

In [ ]:
minute_files_v3 = sorted(
    MINUTE_DATA_DIR.glob("*.parquet")
)

print("Minute files found:", len(minute_files_v3))

assert len(minute_files_v3) > 0, (
    f"No parquet files found in {MINUTE_DATA_DIR}"
)

minute_files_v3[:5]

In [ ]:
test_minute_file = minute_files_v3[0]
test_symbol = test_minute_file.stem

test_minute_raw_df = pd.read_parquet(
    test_minute_file
)

test_minute_features_df = (
    create_direction_v3_minute_features(
        minute_df=test_minute_raw_df,
        symbol=test_symbol,
    )
)

print("Test symbol:", test_symbol)
print("Raw minute shape:", test_minute_raw_df.shape)
print(
    "Daily minute-feature shape:",
    test_minute_features_df.shape,
)

test_minute_features_df.head()

In [ ]:
assert not test_minute_features_df.empty

assert not test_minute_features_df.duplicated(
    ["symbol", "pred_date"]
).any()

bounded_zero_one_features = [
    "intraday_path_efficiency",
    "intraday_up_minute_fraction",
    "late_session_volatility_share_60m",
    "early_session_volatility_share_60m",
    "largest_minute_move_share",
    "closing_volume_share_60m",
    "close_location_in_range",
    "volume_concentration_hhi",
    "minute_bar_coverage",
]

for column in bounded_zero_one_features:
    assert test_minute_features_df[
        column
    ].dropna().between(0, 1.000001).all(), column

assert test_minute_features_df[
    "signed_closing_volume_pressure"
].dropna().between(-1.000001, 1.000001).all()

non_negative_features = [
    "intraday_realized_volatility",
    "intraday_high_low_range_pct",
]

for column in non_negative_features:
    assert (
        test_minute_features_df[column]
        .dropna()
        .ge(0)
        .all()
    ), column

assert not np.isinf(
    test_minute_features_df[
        V3_MINUTE_FEATURE_COLUMNS
    ].to_numpy(dtype=float)
).any()

print("Single-symbol minute validation passed.")

test_minute_features_df[
    V3_MINUTE_FEATURE_COLUMNS
].describe().T

In [ ]:
from time import perf_counter

minute_feature_frames = []
minute_processing_failures = []

start_time = perf_counter()

for file_number, minute_file in enumerate(
    minute_files_v3,
    start=1,
):
    symbol = minute_file.stem

    try:
        minute_raw_df = pd.read_parquet(minute_file)

        symbol_minute_features = (
            create_direction_v3_minute_features(
                minute_df=minute_raw_df,
                symbol=symbol,
            )
        )

        minute_feature_frames.append(
            symbol_minute_features
        )

    except Exception as exc:
        minute_processing_failures.append(
            {
                "symbol": symbol,
                "file": str(minute_file),
                "error": repr(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 10 == 0
        or file_number == len(minute_files_v3)
    ):
        elapsed_minutes = (
            perf_counter() - start_time
        ) / 60

        print(
            f"Processed {file_number}/{len(minute_files_v3)} "
            f"files | failures: {len(minute_processing_failures)} "
            f"| elapsed: {elapsed_minutes:.1f} minutes"
        )

elapsed_minutes = (
    perf_counter() - start_time
) / 60

print("\nMinute processing complete.")
print("Successful files:", len(minute_feature_frames))
print("Failed files:", len(minute_processing_failures))
print(f"Runtime: {elapsed_minutes:.1f} minutes")

In [ ]:
if minute_processing_failures:
    minute_failure_df = pd.DataFrame(
        minute_processing_failures
    )

    display(minute_failure_df)
else:
    print("No minute-file processing failures.")

In [ ]:
direction_v3_minute_df = pd.concat(
    minute_feature_frames,
    ignore_index=True,
)

direction_v3_minute_df = (
    direction_v3_minute_df
    .sort_values(["symbol", "pred_date"])
    .reset_index(drop=True)
)

direction_v3_minute_df["pred_date"] = pd.to_datetime(
    direction_v3_minute_df["pred_date"]
)

print("Minute feature shape:", direction_v3_minute_df.shape)
print(
    "Symbols:",
    direction_v3_minute_df["symbol"].nunique(),
)
print(
    "Date range:",
    direction_v3_minute_df["pred_date"].min(),
    "to",
    direction_v3_minute_df["pred_date"].max(),
)

In [ ]:
assert (
    direction_v3_minute_df["symbol"].nunique()
    == len(minute_files_v3)
)

assert not direction_v3_minute_df.duplicated(
    ["symbol", "pred_date"]
).any()

bounded_zero_one_features = [
    "intraday_path_efficiency",
    "intraday_up_minute_fraction",
    "late_session_volatility_share_60m",
    "early_session_volatility_share_60m",
    "largest_minute_move_share",
    "closing_volume_share_60m",
    "close_location_in_range",
    "volume_concentration_hhi",
    "minute_bar_coverage",
]

for column in bounded_zero_one_features:
    assert direction_v3_minute_df[
        column
    ].dropna().between(
        0,
        1.000001,
    ).all(), column

assert direction_v3_minute_df[
    "signed_closing_volume_pressure"
].dropna().between(
    -1.000001,
    1.000001,
).all()

for column in [
    "intraday_realized_volatility",
    "intraday_high_low_range_pct",
]:
    assert direction_v3_minute_df[
        column
    ].dropna().ge(0).all(), column

assert not np.isinf(
    direction_v3_minute_df[
        V3_MINUTE_FEATURE_COLUMNS
    ].to_numpy(dtype=float)
).any()

print("Full V3 minute validation passed.")

In [ ]:
direction_v3_minute_df.to_parquet(
    V3_MINUTE_FEATURES_PATH,
    index=False,
)

print("Saved:", V3_MINUTE_FEATURES_PATH)
print("Shape:", direction_v3_minute_df.shape)

In [ ]:
direction_v3_daily_df["pred_date"] = pd.to_datetime(
    direction_v3_daily_df["pred_date"]
)

direction_v3_minute_df["pred_date"] = pd.to_datetime(
    direction_v3_minute_df["pred_date"]
)

print(
    "Daily duplicate keys:",
    direction_v3_daily_df.duplicated(
        ["symbol", "pred_date"]
    ).sum(),
)

print(
    "Minute duplicate keys:",
    direction_v3_minute_df.duplicated(
        ["symbol", "pred_date"]
    ).sum(),
)

In [ ]:
direction_v3_model_panel = (
    direction_v3_daily_df.merge(
        direction_v3_minute_df,
        on=["symbol", "pred_date"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

print(
    direction_v3_model_panel["_merge"]
    .value_counts(dropna=False)
)

print(
    "Final panel shape:",
    direction_v3_model_panel.shape,
)

In [ ]:
direction_v3_model_panel[
    "minute_features_available"
] = (
    direction_v3_model_panel["_merge"]
    .eq("both")
    .astype("int8")
)

direction_v3_model_panel = (
    direction_v3_model_panel
    .drop(columns="_merge")
)

print(
    direction_v3_model_panel[
        "minute_features_available"
    ].value_counts(dropna=False)
)

In [ ]:
assert len(direction_v3_model_panel) == len(
    direction_v3_daily_df
)

assert not direction_v3_model_panel.duplicated(
    ["symbol", "pred_date"]
).any()

assert set(V3_MINUTE_FEATURE_COLUMNS).issubset(
    direction_v3_model_panel.columns
)

print(
    "Minute feature missingness:"
)

print(
    direction_v3_model_panel[
        V3_MINUTE_FEATURE_COLUMNS
    ].isna().mean().sort_values(
        ascending=False
    )
)

print("\nFinal V3 panel validation passed.")

In [ ]:
direction_v3_model_panel.to_parquet(
    V3_MODEL_PANEL_PATH,
    index=False,
)

print("Saved final V3 panel:")
print(V3_MODEL_PANEL_PATH)
print("Shape:", direction_v3_model_panel.shape)
print(
    "Columns:",
    direction_v3_model_panel.shape[1],
)